In [1]:
from bs4 import BeautifulSoup, Tag
from urllib.parse import urljoin
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
BASE_URL = "https://en.numista.com"
url = "https://en.numista.com/catalogue/pays.php"

In [3]:
def get_page_html(url: str) -> str:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1600,1400")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        driver.get(url)
        html = driver.page_source
    finally:
        driver.quit()

    return html

In [4]:
def get_direct_link(li: Tag):
    for child in li.children:
        if isinstance(child, Tag) and child.name == "a" and "name" in (child.get("class") or []):
            return child
    return None

In [5]:
def get_direct_alt_names(li: Tag):
    for child in li.children:
        if isinstance(child, Tag) and child.name == "span" and "alt_names" in (child.get("class") or []):
            return child.get_text(" ", strip=True)
    return None

In [6]:
def get_child_uls(li: Tag):
    uls = []
    for child in li.children:
        if isinstance(child, Tag):
            if child.name == "ul":
                uls.append(child)
            elif child.name == "details":
                for dchild in child.children:
                    if isinstance(dchild, Tag) and dchild.name == "ul":
                        uls.append(dchild)
    return uls

In [7]:
def walk_tree(li: Tag, path=None):
    if path is None:
        path = []

    rows = []

    a = get_direct_link(li)
    if not a:
        return rows

    name = a.get_text(" ", strip=True)
    href = a.get("href")
    alt_names = get_direct_alt_names(li)

    current_path = path + [name]

    rows.append({
        "name": name,
        "url": urljoin(BASE_URL, href),
        "relative_url": href,
        "alt_names": alt_names,
        "path": " > ".join(current_path),
        "depth": len(current_path) - 1,
        "li_classes": " ".join(li.get("class", [])),
    })

    for ul in get_child_uls(li):
        for child_li in ul.find_all("li", recursive=False):
            rows.extend(walk_tree(child_li, current_path))

    return rows

In [8]:
def scrape_liste_pays(page_url: str) -> pd.DataFrame:
    html = get_page_html(page_url)
    soup = BeautifulSoup(html, "html.parser")

    rows = []

    root_ul = soup.find("ul", class_="liste_pays")

    for li in root_ul.find_all("li", recursive=False):
        rows.extend(walk_tree(li))

    df = pd.DataFrame(rows).drop_duplicates(subset=["url", "path"]).reset_index(drop=True)
    return df

In [9]:
if __name__ == "__main__":
    df = scrape_liste_pays(url)

    print(df.head(20))
    print("Total rows:", len(df))

    df.to_csv("numista_issuers_tree.csv", index=False)

                     name                                                url  \
0             Afghanistan  https://en.numista.com/catalogue/afghanistan_s...   
1           Afghan States  https://en.numista.com/catalogue/afghan_states...   
2           Afghan Cities  https://en.numista.com/catalogue/afghan_cities...   
3     Badakhshan, City of  https://en.numista.com/catalogue/badakhshan_ci...   
4          Balkh, City of  https://en.numista.com/catalogue/balkh_city-1....   
5       Ghaznayn, City of  https://en.numista.com/catalogue/ghaznayn_city...   
6          Herat, City of  https://en.numista.com/catalogue/herat_city-1....   
7      Jalalabad, City of  https://en.numista.com/catalogue/jalalabad_cit...   
8          Kabul, City of  https://en.numista.com/catalogue/kabul_city-1....   
9       Khanabad, City of  https://en.numista.com/catalogue/khanabad_city...   
10      Peshawar, City of  https://en.numista.com/catalogue/peshawar_city...   
11      Qandahar, City of  https://en.nu